In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd


PROJECT_ROOT = Path("/Users/ydnkka/Desktop/PhD Project/projects/scotland")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils import load_analysis_columns, load_pairwise_edges  # noqa: E402


In [3]:
nodes = load_analysis_columns(
    columns=[
        "window_idx",
        "window_id",
        "clade",
        "sex",
        "age_band",
        "age_midpoint",
        "age_group",
        "dz_local_authority",
        "dz_health_board",
        "dz_urban_rural_class",
        "dz_simd_quintile",
        "policy_period",
        "policy_era"
    ],
    window_stride=3,
    renumber_windows=False,
)

nodes.shape

(322214, 17)

In [8]:
(nodes.groupby(["pango_lineage", "window_idx"]).size().sort_values(ascending=False)>=6).sum()

np.int64(1125)

In [9]:
nodes.groupby(["pango_lineage", "window_idx"]).size().sort_values(ascending=False)

pango_lineage  window_idx
BA.2           88            25040
               85            14043
BA.1.1         85            11887
BA.2           94            11779
AY.4           61            11063
                             ...  
BF.9           109               1
               112               1
               118               1
BG.2           94                1
A.23.1         25                1
Length: 2805, dtype: int64

In [ ]:
edges = load_pairwise_edges(windows=34, pango_lineages="B.1.1.7")

print(f"{len(nodes):,} nodes; {len(edges):,} edges")

In [1]:
def prepare_graph_arrays(
    nodes,
    edges,
    node_id_col="sequence_id",
    source_col="id1",
    target_col="id2",
    weight_col="epilink_compatibility",
):
    """
    Convert node and edge dataframes into efficient NumPy arrays.

    Returns
    -------
    node_ids : ndarray
        Node IDs in internal ordering.
    i, j : int32 ndarray
        Internal integer node indices for edge endpoints.
    w : float64 ndarray
        Edge weights.
    id_to_idx : dict
        Mapping from original node ID to internal integer index.
    """

    # Ensure nodes have unique node IDs
    if nodes[node_id_col].duplicated().any():
        raise ValueError("Duplicate node IDs found in nodes dataframe.")

    node_ids = nodes[node_id_col].to_numpy()
    id_to_idx = pd.Series(np.arange(len(nodes), dtype=np.int32), index=node_ids)

    # Map edge source/target IDs to internal integer indices
    i = edges[source_col].map(id_to_idx).to_numpy()
    j = edges[target_col].map(id_to_idx).to_numpy()

    # Check for edges referencing missing nodes
    if pd.isna(i).any() or pd.isna(j).any():
        raise ValueError("Some edges reference node IDs not present in nodes.")

    i = i.astype(np.int32)
    j = j.astype(np.int32)

    # Edge weights
    if weight_col is None:
        w = np.ones(len(edges), dtype=np.float64)
    else:
        w = edges[weight_col].to_numpy(dtype=np.float64)

    # Optional: remove self-loops if present
    mask = i != j
    i = i[mask]
    j = j[mask]
    w = w[mask]

    # Optional: check positive weights
    if np.any(w < 0):
        raise ValueError(
            "Negative edge weights found. Assortativity usually assumes non-negative weights."
        )

    return node_ids, i, j, w, id_to_idx

def weighted_numeric_assortativity(i, j, w, x):
    """
    Weighted assortativity for a scalar node attribute x.

    Parameters
    ----------
    i, j : ndarray
        Edge endpoint indices.
    w : ndarray
        Edge weights.
    x : ndarray
        Numeric node attribute, aligned to internal node index.

    Returns
    -------
    r : float
        Weighted numeric assortativity.
    """

    x = np.asarray(x, dtype=np.float64)

    total_weight_directed = 2.0 * np.sum(w)

    sx = np.sum(w * (x[i] + x[j]))
    sxx = np.sum(w * (x[i] ** 2 + x[j] ** 2))
    sxy = 2.0 * np.sum(w * x[i] * x[j])

    mean_x = sx / total_weight_directed
    var_x = sxx / total_weight_directed - mean_x ** 2
    cov_xy = sxy / total_weight_directed - mean_x ** 2

    if var_x <= 0:
        return np.nan

    return cov_xy / var_x

def weighted_categorical_assortativity(i, j, w, categories, levels=None):
    """
    Weighted assortativity for categorical node attributes.

    Parameters
    ----------
    i, j : ndarray
        Edge endpoint indices.
    w : ndarray
        Edge weights.
    categories : array-like
        Categorical node attribute, aligned to internal node index.
    levels : array-like, optional
        Ordered list of category levels. If not provided, levels are inferred from `categories`.

    Returns
    -------
    r : float
        Weighted categorical assortativity.
    mixing_matrix : ndarray
        Weighted normalized mixing matrix.
    labels : ndarray
        Category labels corresponding to matrix rows/columns.
    """

    codes, labels = pd.factorize(categories, sort=True)

    if np.any(codes < 0):
        raise ValueError("Missing categorical values found. Please impute or filter first.")

    k = len(labels)

    # Weighted counts for observed edge directions
    flat_index = codes[i] * k + codes[j]

    e = np.bincount(
        flat_index,
        weights=w,
        minlength=k * k
    ).reshape(k, k)

    # Since graph is undirected, count both directions
    e = e + e.T

    # Normalize to proportions
    e = e / e.sum()

    a = e.sum(axis=1)

    expected_same = np.sum(a ** 2)
    observed_same = np.trace(e)

    denominator = 1.0 - expected_same

    if denominator <= 0:
        r = np.nan
    else:
        r = (observed_same - expected_same) / denominator
    
    e, labels = complete_mixing_matrix(
        e,
        group_labels=labels,
        all_categories=levels if levels is not None else labels,
        fill_value=0.0,
        return_dataframe=True,
    )

    return r, e, labels

def complete_mixing_matrix(
    matrix,
    group_labels,
    all_categories,
    fill_value=0.0,
    return_dataframe=True,
):
    """
    Expand a mixing matrix so that it includes all requested categories.

    Parameters
    ----------
    matrix : array-like, shape (k, k)
        Existing mixing matrix. Rows/columns correspond to `group_labels`.
    group_labels : array-like, length k
        Labels currently represented in `matrix`.
    all_categories : array-like, length K
        Full list of desired categories. Output matrix will use this order.
        Categories absent from `group_labels` get zero rows/columns.
    fill_value : float, default 0.0
        Value to use for missing row/column entries.
    return_dataframe : bool, default True
        If True, return a pandas DataFrame with labelled rows/columns.
        If False, return a NumPy array and the output labels.

    Returns
    -------
    completed : ndarray or DataFrame
        Completed mixing matrix of shape (K, K).
    labels : ndarray, optional
        Returned only if `return_dataframe=False`.
    """

    matrix = np.asarray(matrix)
    group_labels = np.asarray(group_labels)
    all_categories = np.asarray(all_categories)

    if matrix.ndim != 2 or matrix.shape[0] != matrix.shape[1]:
        raise ValueError("`matrix` must be a square 2D array.")

    if matrix.shape[0] != len(group_labels):
        raise ValueError(
            "`matrix` shape does not match length of `group_labels`."
        )

    if len(pd.Index(group_labels)) != len(pd.Index(group_labels).unique()):
        raise ValueError("`group_labels` contains duplicates.")

    if len(pd.Index(all_categories)) != len(pd.Index(all_categories).unique()):
        raise ValueError("`all_categories` contains duplicates.")

    extra_labels = set(group_labels) - set(all_categories)
    if extra_labels:
        raise ValueError(
            f"`group_labels` contains labels not present in `all_categories`: "
            f"{sorted(extra_labels)}"
        )

    completed = np.full(
        (len(all_categories), len(all_categories)),
        fill_value,
        dtype=matrix.dtype,
    )

    label_to_pos = {label: pos for pos, label in enumerate(all_categories)}

    old_positions = np.array(
        [label_to_pos[label] for label in group_labels],
        dtype=np.int64,
    )

    completed[np.ix_(old_positions, old_positions)] = matrix

    if return_dataframe:
        return pd.DataFrame(
            completed,
            index=all_categories,
            columns=all_categories,
        ), all_categories

    return completed, all_categories

def multiplier_bootstrap(
    stat_fn,
    w,
    B=500,
    alpha=0.05,
    seed=123,
    chunk_size=None,
):
    """
    Multiplier/Bayesian bootstrap for an edge-weighted statistic.

    Parameters
    ----------
    stat_fn : callable
        Function accepting a weight vector and returning a scalar statistic.
    w : ndarray
        Original edge weights.
    B : int
        Number of bootstrap replicates.
    alpha : float
        For 95% CI, use alpha=0.05.
    seed : int
        Random seed.
    chunk_size : ignored here
        Placeholder if you later want chunked generation.

    Returns
    -------
    point : float
        Statistic using original weights.
    ci : tuple
        Percentile confidence interval.
    se : float
        Bootstrap standard error.
    boot : ndarray
        Bootstrap replicate statistics.
    """

    rng = np.random.default_rng(seed)

    point = stat_fn(w)
    boot = np.empty(B, dtype=np.float64)

    for b in range(B):
        # Exponential(1) multipliers
        g = rng.exponential(scale=1.0, size=len(w))
        boot[b] = stat_fn(w * g)

    lo, hi = np.quantile(boot, [alpha / 2, 1 - alpha / 2])
    se = boot.std(ddof=1)

    return point, (lo, hi), se, boot

def node_strengths(i, j, w, n_nodes):
    """
    Weighted degree / node strength for an undirected graph.
    """

    strength = np.zeros(n_nodes, dtype=np.float64)

    np.add.at(strength, i, w)
    np.add.at(strength, j, w)

    return strength

In [ ]:
node_ids, i, j, w, id_to_idx = prepare_graph_arrays(
    nodes,
    edges,
)

In [ ]:
x_age = nodes["age_midpoint"].to_numpy(dtype=np.float64)

r_age = weighted_numeric_assortativity(i, j, w, x_age)
print(r_age)

In [ ]:
r_age_group, mix_age_group, age_group_labels = weighted_categorical_assortativity(
    i,
    j,
    w,
    nodes["age_group"]
)

print(r_age_group)

In [ ]:
strength = node_strengths(i, j, w, len(nodes))

r_strength = weighted_numeric_assortativity(i, j, w, strength)
print(r_strength)

In [ ]:
x_age = nodes["age_midpoint"].to_numpy(dtype=np.float64)

point, ci, se, boot = multiplier_bootstrap(
    stat_fn=lambda wb: weighted_numeric_assortativity(i, j, wb, x_age),
    w=w,
    B=500,
    alpha=0.05,
    seed=123,
)

print(f"Weighted age assortativity: {point:.4f}")
print(f"SE: {se:.4f}")
print(f"95% CI: [{ci[0]:.4f}, {ci[1]:.4f}]")

In [ ]:
city_values = nodes["age_group"].to_numpy()

point, ci, se, boot = multiplier_bootstrap(
    stat_fn=lambda wb: weighted_categorical_assortativity(i, j, wb, city_values)[0],
    w=w,
    B=500,
    alpha=0.05,
    seed=123,
)

print(f"Weighted city assortativity: {point:.4f}")
print(f"SE: {se:.4f}")
print(f"95% CI: [{ci[0]:.4f}, {ci[1]:.4f}]")

In [ ]:
strength = node_strengths(i, j, w, len(nodes))

point, ci, se, boot = multiplier_bootstrap(
    stat_fn=lambda wb: weighted_numeric_assortativity(i, j, wb, strength),
    w=w,
    B=500,
)

print(f"Weighted strength assortativity: {point:.4f}")
print(f"SE: {se:.4f}")
print(f"95% CI: [{ci[0]:.4f}, {ci[1]:.4f}]")

In [ ]:
def strength_assortativity_recomputed(i, j, wb, n_nodes):
    sb = node_strengths(i, j, wb, n_nodes)
    return weighted_numeric_assortativity(i, j, wb, sb)

point, ci, se, boot = multiplier_bootstrap(
    stat_fn=lambda wb: strength_assortativity_recomputed(i, j, wb, len(nodes)),
    w=w,
    B=500,
)

print(f"Weighted strength assortativity: {point:.4f}")
print(f"SE: {se:.4f}")
print(f"95% CI: [{ci[0]:.4f}, {ci[1]:.4f}]")

You could report the result like this:

Weighted assortativity was computed from the undirected weighted edge list using the weighted mixing matrix for categorical attributes and weighted endpoint correlation for numeric attributes. Uncertainty was estimated using a multiplier bootstrap over edges with 500 replicates.

Or for strength:

Weighted degree assortativity was computed as the weighted Pearson correlation of endpoint node strengths, with node strengths recomputed within each bootstrap replicate.